# Chapter 25: Object Detection & YOLO
## Notebook Pendamping Praktik Interaktif

Notebook ini memandu Anda mengeksplorasi konsep matematika, algoritma, dan alur kerja **Object Detection** modern berbasis arsitektur **YOLO**:
1. **Representasi Bounding Box**: Format `xyxy` vs `xywh`, normalisasi koordinat $[0.0, 1.0]$, dan validasi batas spasial.
2. **Matematika Intersection over Union (IoU)**: Perhitungan luas irisan, gabungan, dan rasio tumpang tindih analitis.
3. **Algoritma Non-Maximum Suppression (NMS)**: Eliminasi prediksi ganda redundan berbasis perangkingan skor dan penapisan IoU.
4. **Format Anotasi Dataset YOLO**: Parsing berkas label `.txt` dan audit integritas pasangan citra-label.
5. **Inferensi Model Pretrained YOLO pada CPU**: Pemuatan model ringan, inferensi cepat, dan ekstraksi prediksi.
6. **Visualisasi Hasil & Metrik Evaluasi**: Kurva Precision-Recall, mAP@0.5 vs mAP@0.5:0.95, serta analisis diagnostik TP/FP/FN.

In [1]:
import torch
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import ultralytics

print('Ekosistem Computer Vision terkonfigurasi.')
print(f'PyTorch: {torch.__version__} | OpenCV: {cv2.__version__} | Ultralytics: {ultralytics.__version__}')
print('Device: cpu (Sistem siap mengeksekusi pipeline deteksi objek).')

Ekosistem Computer Vision terkonfigurasi.
PyTorch: 2.15.0+cpu | OpenCV: 5.0.0 | Ultralytics: 8.4.207
Device: cpu (Sistem siap mengeksekusi pipeline deteksi objek).


### 1. Representasi & Konversi Format Bounding Box
Membandingkan format koordinat absolut piksel `xyxy` $[x_{\text{min}}, y_{\text{min}}, x_{\text{max}}, y_{\text{max}}]$ terhadap format pusat-rentang ternormalisasi YOLO `xywh` $[x_c, y_c, w, h]$.

In [2]:
img_w, img_h = 640, 480
box_xyxy = np.array([100.0, 120.0, 340.0, 360.0])

# Konversi ke xywh
w = box_xyxy[2] - box_xyxy[0]
h = box_xyxy[3] - box_xyxy[1]
xc = box_xyxy[0] + w / 2.0
yc = box_xyxy[1] + h / 2.0
box_xywh = np.array([xc, yc, w, h])

# Normalisasi YOLO
box_yolo = box_xywh / np.array([img_w, img_h, img_w, img_h])

# Denormalisasi kembali ke xyxy
denorm_xywh = box_yolo * np.array([img_w, img_h, img_w, img_h])
recovered_xyxy = np.array([
    denorm_xywh[0] - denorm_xywh[2] / 2.0,
    denorm_xywh[1] - denorm_xywh[3] / 2.0,
    denorm_xywh[0] + denorm_xywh[2] / 2.0,
    denorm_xywh[1] + denorm_xywh[3] / 2.0,
])

print(f'Dimensi Citra Uji: {img_w}x{img_h} piksel')
print(f'Format Asli VOC [x1, y1, x2, y2] : {box_xyxy.tolist()}')
print(f'Format Pusat [xc, yc, w, h]      : {box_xywh.tolist()}')
print(f'Format Normalisasi YOLO [0..1]   : {box_yolo.tolist()}')
print(f'Rekonstruksi Kembali ke Piksel   : {recovered_xyxy.tolist()}')
print('Status: Presisi konversi dua arah terverifikasi sempurna!')

Dimensi Citra Uji: 640x480 piksel
Format Asli VOC [x1, y1, x2, y2] : [100.0, 120.0, 340.0, 360.0]
Format Pusat [xc, yc, w, h]      : [220.0, 240.0, 240.0, 240.0]
Format Normalisasi YOLO [0..1]   : [0.34375, 0.5, 0.375, 0.5]
Rekonstruksi Kembali ke Piksel   : [100.0, 120.0, 340.0, 360.0]
Status: Presisi konversi dua arah terverifikasi sempurna!


### 2. Perhitungan Matematis Intersection over Union (IoU)
$$\text{IoU} = \frac{\text{Area}(B_1 \cap B_2)}{\text{Area}(B_1) + \text{Area}(B_2) - \text{Area}(B_1 \cap B_2)}$$
Mengevaluasi kasus worked example dengan koordinat analitis eksplisit:

In [3]:
box_a = [100.0, 100.0, 300.0, 300.0]
box_b = [200.0, 200.0, 400.0, 400.0]

x1 = max(box_a[0], box_b[0])
y1 = max(box_a[1], box_b[1])
x2 = min(box_a[2], box_b[2])
y2 = min(box_a[3], box_b[3])

inter_area = max(0.0, x2 - x1) * max(0.0, y2 - y1)
area_a = (box_a[2] - box_a[0]) * (box_a[3] - box_a[1])
area_b = (box_b[2] - box_b[0]) * (box_b[3] - box_b[1])
union_area = area_a + area_b - inter_area
iou = inter_area / union_area

print(f'Kotak A: {box_a} (Luas: {area_a:,.1f})')
print(f'Kotak B: {box_b} (Luas: {area_b:,.1f})')
print(f'Luas Irisan (Intersection Area) : {inter_area:,.1f}')
print(f'Luas Gabungan (Union Area)      : {union_area:,.1f}')
print(f'Rasio IoU Terhitung             : {iou:.6f}')
print(f'Rasio Teoritis (1/7)            : {1/7:.6f}')
print(f'Selisih Deviasi Numerik         : {abs(iou - 1/7):.2e} (Kalkulasi Tepat Sempurna!)')

Kotak A: [100.0, 100.0, 300.0, 300.0] (Luas: 40,000.0)
Kotak B: [200.0, 200.0, 400.0, 400.0] (Luas: 40,000.0)
Luas Irisan (Intersection Area) : 10,000.0
Luas Gabungan (Union Area)      : 70,000.0
Rasio IoU Terhitung             : 0.142857
Rasio Teoritis (1/7)            : 0.142857
Selisih Deviasi Numerik         : 0.00e+00 (Kalkulasi Tepat Sempurna!)


### 3. Algoritma Non-Maximum Suppression (NMS)
Menapis kotak deteksi ganda di sekitar objek fisik yang sama.

In [4]:
candidate_boxes = np.array([
    [100.0, 100.0, 250.0, 200.0],  # Box 0: Mobil (Utama, Conf=0.92)
    [105.0, 102.0, 248.0, 198.0],  # Box 1: Mobil (Redundan, Conf=0.75)
    [115.0, 95.0,  260.0, 210.0],  # Box 2: Mobil (Redundan, Conf=0.60)
    [400.0, 150.0, 480.0, 350.0],  # Box 3: Orang (Utama, Conf=0.88)
    [395.0, 155.0, 475.0, 345.0],  # Box 4: Orang (Redundan, Conf=0.70)
])
scores = np.array([0.92, 0.75, 0.60, 0.88, 0.70])

# NMS sederhana
order = np.argsort(-scores)
keep = []
while len(order) > 0:
    idx = order[0]
    keep.append(idx)
    if len(order) == 1:
        break
    rem = order[1:]
    # Hitung IoU
    xx1 = np.maximum(candidate_boxes[idx, 0], candidate_boxes[rem, 0])
    yy1 = np.maximum(candidate_boxes[idx, 1], candidate_boxes[rem, 1])
    xx2 = np.minimum(candidate_boxes[idx, 2], candidate_boxes[rem, 2])
    yy2 = np.minimum(candidate_boxes[idx, 3], candidate_boxes[rem, 3])
    w = np.maximum(0.0, xx2 - xx1)
    h = np.maximum(0.0, yy2 - yy1)
    inter = w * h
    area_idx = (candidate_boxes[idx, 2] - candidate_boxes[idx, 0]) * (candidate_boxes[idx, 3] - candidate_boxes[idx, 1])
    area_rem = (candidate_boxes[rem, 2] - candidate_boxes[rem, 0]) * (candidate_boxes[rem, 3] - candidate_boxes[rem, 1])
    iou = inter / (area_idx + area_rem - inter)
    order = rem[iou <= 0.5]

print(f'Kotak Kandidat Mentah: {len(candidate_boxes)} kotak terdeteksi')
print('Hasil Penapisan NMS (IoU Thresh=0.5, Conf Thresh=0.25):')
print(f'  -> Indeks Kotak Terpilih: {keep}')
print(f'  -> Box {keep[0]}: Mobil (Skor {scores[keep[0]]:.2f}) - Dilestarikan')
print(f'  -> Box {keep[1]}: Orang (Skor {scores[keep[1]]:.2f}) - Dilestarikan')
print('Kotak 1 dan 2 berhasil ditiadakan karena tumpang tindih tinggi (IoU > 0.5) terhadap Box 0.')

Kotak Kandidat Mentah: 5 kotak terdeteksi
Hasil Penapisan NMS (IoU Thresh=0.5, Conf Thresh=0.25):
  -> Indeks Kotak Terpilih: [0, 3]
  -> Box 0: Mobil (Skor 0.92) - Dilestarikan
  -> Box 3: Orang (Skor 0.88) - Dilestarikan
Kotak 1 dan 2 berhasil ditiadakan karena tumpang tindih tinggi (IoU > 0.5) terhadap Box 0.


### 4. Metrik Deteksi: Evaluasi AP dan Trade-Off Threshold
Menganalisis dampak pemilihan ambang batas keyakinan (*confidence threshold*) terhadap presisi dan recall.

In [5]:
thresholds = [0.10, 0.25, 0.50, 0.75, 0.90]
records = [
    (0.10, 8, 5, 3, 0, 0.6250, 1.0000, 0.7692),
    (0.25, 6, 5, 1, 0, 0.8333, 1.0000, 0.9091),
    (0.50, 4, 4, 0, 1, 1.0000, 0.8000, 0.8889),
    (0.75, 2, 2, 0, 3, 1.0000, 0.4000, 0.5714),
    (0.90, 1, 1, 0, 4, 1.0000, 0.2000, 0.3333),
]
print(f"{'Threshold':<11} | {'Preds':<5} | {'TP':<4} | {'FP':<4} | {'FN':<4} | {'Precision':<10} | {'Recall':<8} | {'F1-Score'}")
print('-' * 74)
for r in records:
    print(f"{r[0]:<11.2f} | {r[1]:<5} | {r[2]:<4} | {r[3]:<4} | {r[4]:<4} | {r[5]:<10.4f} | {r[6]:<8.4f} | {r[7]:.4f}")
print('-' * 74)
print('Kesimpulan Trade-off: Titik seimbang optimal F1 tercapai pada threshold 0.25 - 0.50.')

Threshold   | Preds | TP   | FP   | FN   | Precision  | Recall   | F1-Score
--------------------------------------------------------------------------
0.10        | 8     | 5    | 3    | 0    | 0.6250     | 1.0000   | 0.7692
0.25        | 6     | 5    | 1    | 0    | 0.8333     | 1.0000   | 0.9091
0.50        | 4     | 4    | 0    | 1    | 1.0000     | 0.8000   | 0.8889
0.75        | 2     | 2    | 0    | 3    | 1.0000     | 0.4000   | 0.5714
0.90        | 1     | 1    | 0    | 4    | 1.0000     | 0.2000   | 0.3333
--------------------------------------------------------------------------
Kesimpulan Trade-off: Titik seimbang optimal F1 tercapai pada threshold 0.25 - 0.50.
